# Qwen3-VL: APTOS training and APTOS / Messidor testing

This notebook orchestrates the repository scripts using the Python executable of the active
**Anaconda `afterimage_env` kernel**. It uses every row of the supplied CSV in its assigned split:

| Role | APTOS | Messidor | Total |
| --- | ---: | ---: | ---: |
| Train | 2,929 | 0 | 2,929 |
| Validation | 366 | 0 | 366 |
| Test | 367 | 1,057 | 1,424 |

**Messidor remains an external test set.** There is no re-splitting and no training on test images.
Validation loss selects the best checkpoint; test results must not guide checkpoint selection.
The existing trainer balances training classes with replacement, so all training rows are eligible,
but a balanced epoch does not promise to visit each image exactly once.

The images are already cropped, padded, resized to 512, and processed with CLAHE. Do not apply CLAHE again.
The CSV's label-bearing `vlm_prompt` captions are excluded from exported inputs.

1. Select the Conda kernel and check configuration.
2. Validate/export the existing splits without modifying the source CSV or images.
3. Train via `train_dr_qwen3vl_runsave.py` and select that attempt's saved `best_lora`.
4. Test via `eval_dataset_dr_qwen3vl_selectrun.py`, which imports generation/parsing from
   `eval_infer_dr_qwen3vl.py`.
5. Save combined and separate APTOS/Messidor predictions, metrics, and confusion matrices.

Full testing generates one answer for every test image. Source-specific reports use those same
predictions, avoiding a second pass over 1,424 images. Optional independent source evaluations are provided.

## 1. Select the Anaconda environment

In **Anaconda Prompt**, from the repository root:

```bat
conda activate afterimage_env
python -m pip install ipykernel
python -m ipykernel install --user --name afterimage_env --display-name "Python (afterimage_env)"
```

Open this notebook in VS Code or your Jupyter client and select **Python (afterimage_env)**.
If you need a Jupyter client, install `notebook` separately or use VS Code's Jupyter extension.
The project model stack should already be installed using the repository setup instructions;
this notebook does not upgrade Torch, Unsloth, NumPy, or pandas. Scikit-learn is optional:
the evaluator has a NumPy fallback and the reports below use NumPy directly.

Stop/unload the app's model servers before training so they release GPU memory. Full training and
testing run only when you execute the relevant cells; they can take hours. Reuse an existing run
by setting `RUN_PATH_OVERRIDE` below. The trainer does not implement interrupted-checkpoint resume.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata as metadata
import json
import os
import subprocess
import sys
import uuid

import numpy as np
import pandas as pd
from IPython.display import display

# Locate the checkout when launched from either its root or this notebook's folder.
REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "finetuning/Qwen3/train_dr_qwen3vl_runsave.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Start Jupyter from the Afterimage checkout or set REPO_ROOT explicitly.")
QWEN_DIR = REPO_ROOT / "finetuning/Qwen3"
TRAIN_SCRIPT = QWEN_DIR / "train_dr_qwen3vl_runsave.py"
EVAL_SCRIPT = QWEN_DIR / "eval_dataset_dr_qwen3vl_selectrun.py"
INFER_SCRIPT = QWEN_DIR / "eval_infer_dr_qwen3vl.py"

DATA_ROOT = Path(r"C:\Users\moht2\Downloads\DR_Aptos_Messidor_Only_Full_Dataset\Res_512")
CSV_PATH = DATA_ROOT / "Split_80_10_10/dataset_80_10_10.csv"
IMAGE_DIR = DATA_ROOT / "Aptos_messidor_external_CLAHE"
WORK_ROOT = QWEN_DIR / "artifacts/aptos_messidor"

BASE_MODEL = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit"
IMG_SIZE = 512
SEED = 42
EPOCHS = 50                   # Maximum; validation early stopping usually ends sooner.
LEARNING_RATE = 5e-4          # Historical project setting; edit before starting a new experiment.
BATCH_SIZE = 1               # Conservative starting point for the 16 GB GPU.
GRAD_ACCUM = 8               # Effective batch size = 8 on one GPU.
EVAL_BATCH_SIZE = 1
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
MAX_LENGTH = 0               # No token truncation: preserve all image tokens.
EARLY_STOPPING_PATIENCE = 2
VERIFY_IMAGES = True         # Read every PNG to detect corrupt files and unexpected dimensions.

MAX_NEW_TOKENS = 48
TEMPERATURE = 0.0            # Greedy evaluation for repeatable source comparisons.
MIN_P = 0.1                 # Used only when TEMPERATURE > 0.
RUN_EVAL_SMOKE = False       # Optional three-image check before full evaluation.
RUN_SEPARATE_SOURCE_EVAL = False  # Optional extra inference passes; reports are always separate.

# Set this to an existing run folder or its best_lora folder to skip training.
RUN_PATH_OVERRIDE = None
# Example: RUN_PATH_OVERRIDE = Path(r"C:\...\runs\aptos_external_messidor_..._batchsize8")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
ATTEMPT_DIR = WORK_ROOT / "experiments" / RUN_ID
CHECKPOINT_DIR = ATTEMPT_DIR / "checkpoints"
RUNS_ROOT = ATTEMPT_DIR / "runs"
REPORT_DIR = ATTEMPT_DIR / "reports"
LOG_DIR = ATTEMPT_DIR / "logs"

print("Kernel:", sys.executable)
print("Conda environment:", os.environ.get("CONDA_DEFAULT_ENV", "not exported by this notebook client"))
if Path(sys.prefix).name.casefold() != "afterimage_env":
    raise RuntimeError("Select the afterimage_env kernel before continuing. Current prefix: " + sys.prefix)
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("The project's verified environment uses Python 3.11.")
for p in (TRAIN_SCRIPT, EVAL_SCRIPT, INFER_SCRIPT, CSV_PATH):
    if not p.is_file():
        raise FileNotFoundError(p)
if not IMAGE_DIR.is_dir():
    raise FileNotFoundError(IMAGE_DIR)
print("Attempt:", ATTEMPT_DIR)
print("Effective training batch size:", BATCH_SIZE * GRAD_ACCUM)

Kernel: c:\Users\moht2\.conda\envs\afterimage_env\python.exe
Conda environment: not exported by this notebook client
Attempt: c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\aptos_messidor\experiments\20261104T153559Z_bc11a971
Effective training batch size: 8


## 2. Validate all rows and preserve the CSV splits

Missing/corrupt images, invalid labels, duplicate image paths, or patient IDs shared between splits
stop execution. Patient checks use `(source_dataset, patient_id)` so IDs from different datasets do
not collide. These checks establish separation by the provided IDs; they do not detect visually
duplicate images or recover missing clinical patient identities.

In [2]:
source_bytes = CSV_PATH.read_bytes()
source_sha256 = hashlib.sha256(source_bytes).hexdigest()
df = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
required = {"image_path", "label", "patient_id", "source_dataset", "split"}
if not required.issubset(df.columns):
    raise ValueError("Missing CSV columns: " + str(sorted(required - set(df.columns))))
# Explicitly exclude vlm_prompt: its captions reveal the target grade.
df = df[["image_path", "label", "patient_id", "source_dataset", "split"]].copy()
for col in df.columns:
    df[col] = df[col].str.strip()
if df.empty or df.eq("").any().any():
    raise ValueError("The dataset is empty or contains empty required fields.")
df["split"] = df["split"].str.lower()
df["source_dataset"] = df["source_dataset"].str.lower()
if set(df["split"]) != {"train", "val", "test"}:
    raise ValueError("Expected the existing train/val/test split values.")
if set(df["source_dataset"]) != {"aptos", "messidor"}:
    raise ValueError("Expected aptos and messidor source_dataset values.")
if not df["label"].str.fullmatch(r"[0-4]").all():
    raise ValueError("Each label must be an integer 0..4.")
df["label"] = df["label"].astype(int)
df["image_path"] = df["image_path"].map(
    lambda value: str((IMAGE_DIR / Path(value.replace("\\", "/"))).resolve())
)
image_keys = df["image_path"].map(os.path.normcase)
if image_keys.duplicated().any():
    raise ValueError("Duplicate image paths found; resolve duplicates before training.")
patient_splits = df.groupby(["source_dataset", "patient_id"])["split"].nunique()
if patient_splits.gt(1).any():
    raise ValueError("Patient IDs cross dataset splits: " + str(patient_splits[patient_splits.gt(1)].head()))
missing = [p for p in df["image_path"] if not Path(p).is_file()]
if missing:
    raise FileNotFoundError(f"{len(missing)} images missing. Examples: {missing[:5]}")
if not df.loc[df["source_dataset"].eq("messidor"), "split"].eq("test").all():
    raise ValueError("This protocol requires every Messidor row to remain in the external test set.")
if set(df.loc[df["split"].ne("test"), "source_dataset"]) != {"aptos"}:
    raise ValueError("Training and validation must contain only APTOS for this protocol.")

if VERIFY_IMAGES:
    from PIL import Image
    for index, path in enumerate(df["image_path"], 1):
        with Image.open(path) as image:
            if image.size != (IMG_SIZE, IMG_SIZE):
                raise ValueError(f"Unexpected image size {image.size}: {path}")
            image.verify()
        if index % 1000 == 0 or index == len(df):
            print(f"Verified images: {index}/{len(df)}")

split_counts = pd.crosstab(df["split"], df["source_dataset"]).reindex(["train", "val", "test"], fill_value=0)
split_counts["total"] = split_counts.sum(axis=1)
display(split_counts)
display(pd.crosstab([df["split"], df["source_dataset"]], df["label"]).reindex(columns=range(5), fill_value=0))
print(f"Validated {len(df):,} rows. CSV SHA-256: {source_sha256}")

Verified images: 1000/4719
Verified images: 2000/4719
Verified images: 3000/4719
Verified images: 4000/4719
Verified images: 4719/4719


source_dataset,aptos,messidor,total
split,,,
train,2929,0,2929
val,366,0,366
test,367,1057,1424


label                    0    1    2    3    4
split source_dataset                          
test  aptos            180   33   98   21   35
      messidor         468  207  290   71   21
train aptos           1454  302  786  157  230
val   aptos            171   35  115   15   30

Validated 4,719 rows. CSV SHA-256: e90c8846dc7d6eb6b26122d0523d53053441b19dd7fb0334eafce9b854865266


In [3]:
# Content-addressed exports avoid stale trainer caches when the CSV or image root changes.
dataset_key = hashlib.sha256(source_bytes + str(IMAGE_DIR.resolve()).encode() + b"split-export-v1").hexdigest()[:20]
EXPORT_DIR = WORK_ROOT / "data" / dataset_key
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
frames = {split: df.loc[df["split"].eq(split)].reset_index(drop=True) for split in ("train", "val", "test")}
frames["test_aptos"] = frames["test"].loc[frames["test"]["source_dataset"].eq("aptos")].reset_index(drop=True)
frames["test_messidor"] = frames["test"].loc[frames["test"]["source_dataset"].eq("messidor")].reset_index(drop=True)
CSV_FILES = {}
for name, frame in frames.items():
    if frame.empty:
        raise ValueError("Empty split: " + name)
    destination = EXPORT_DIR / f"{name}.csv"
    content = frame.to_csv(index=False, lineterminator="\n")
    if destination.exists() and destination.read_text(encoding="utf-8") != content:
        raise RuntimeError("Existing export differs; inspect it before replacing: " + str(destination))
    destination.write_text(content, encoding="utf-8", newline="")
    CSV_FILES[name] = destination

manifest = {
    "source_csv": str(CSV_PATH), "source_csv_sha256": source_sha256,
    "image_dir": str(IMAGE_DIR.resolve()), "split_policy": "preserve_csv_messidor_external_test",
    "rows": len(df), "patient_ids_disjoint": True, "images_verified": VERIFY_IMAGES,
    "counts": {name: len(frame) for name, frame in frames.items()},
    "csv_files": {name: str(path) for name, path in CSV_FILES.items()},
    "csv_sha256": {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in CSV_FILES.items()},
}
(EXPORT_DIR / "dataset_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Split exports:", EXPORT_DIR)
print("The source CSV and image files have not been modified.")

Split exports: c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\aptos_messidor\data\472ed7fde07b97fcb0b4
The source CSV and image files have not been modified.


## 3. Check the training environment in a subprocess

The kernel does not load a GPU model. Every subprocess uses `sys.executable`, UTF-8 logs and argument
lists that support Windows paths with spaces or `&`. Training runs on one CUDA GPU. If memory is
insufficient, keep batch size at 1 and lower image size only as a new, consistently configured experiment.

In [4]:
def run_live(command, log_path):
    command = [str(part) for part in command]
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    env = {**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false"}
    print(subprocess.list2cmdline(command), flush=True)
    log_path.with_suffix(".command.json").write_text(json.dumps(command, indent=2), encoding="utf-8")
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=REPO_ROOT, env=env, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            return_code = process.wait()
        except BaseException:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            raise
        finally:
            process.stdout.close()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

packages = ("torch", "torchvision", "unsloth", "unsloth_zoo", "transformers", "trl", "peft", "datasets", "pandas", "numpy")
package_versions = {name: metadata.version(name) for name in packages}
display(pd.Series(package_versions, name="installed version"))
preflight = """
import pandas
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from peft import PeftModel
import torch
assert torch.cuda.is_available(), 'CUDA is unavailable in this kernel environment'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print('Training and evaluation imports passed.')
"""
run_live([sys.executable, "-u", "-c", preflight], LOG_DIR / "preflight.log")

torch           2.14.1+cu130
torchvision     0.29.1+cu130
unsloth           2026.10.10
unsloth_zoo       2026.10.10
transformers          5.17.0
trl                   1.15.0
peft                  0.21.2
datasets               4.8.5
pandas                 3.0.6
numpy                  2.4.6
Name: installed version, dtype: str

c:\Users\moht2\.conda\envs\afterimage_env\python.exe -u -c "
import pandas
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from peft import PeftModel
import torch
assert torch.cuda.is_available(), 'CUDA is unavailable in this kernel environment'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print('Training and evaluation imports passed.')
"
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1104 18:36:21.697000 38756 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1104 18:36:21.739000 38756 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and i

## 4. Train and save the best validation checkpoint

`--csv_path` receives **only train.csv**. Both `--val_csv` and `--test_csv` are supplied, so the trainer
does not make a new random split. The test dataset is formatted but is never passed to `SFTTrainer`.
Early stopping uses validation loss; the selected adapter and processor are exported to `best_lora`.

The 50-epoch maximum, learning rate, and effective batch size follow the earlier project notebook.
They are editable experiment settings, not a claim that they are optimal for this dataset.
Each attempt has its own checkpoint, cache, run, report and log locations. Rerunning this cell refuses
to reuse a checkpoint directory. To start over, rerun the configuration cell to generate a new `RUN_ID`.
To evaluate an existing completed run, set `RUN_PATH_OVERRIDE` and rerun the notebook.

In [5]:
TRAIN_COMMAND = [
    sys.executable, "-u", TRAIN_SCRIPT,
    "--csv_path", CSV_FILES["train"], "--val_csv", CSV_FILES["val"], "--test_csv", CSV_FILES["test"],
    "--image_dir", IMAGE_DIR, "--ext", ".png", "--img_size", str(IMG_SIZE),
    "--seed", str(SEED), "--model_name", BASE_MODEL, "--load_in_4bit",
    "--output_dir", CHECKPOINT_DIR, "--runs_root", RUNS_ROOT,
    "--run_name_prefix", "aptos_external_messidor_" + RUN_ID,
    "--cache_dir", ATTEMPT_DIR / "dataset_cache", "--preproc_num_proc", "0",
    "--epochs", str(EPOCHS), "--batch_size", str(BATCH_SIZE), "--eval_batch_size", str(EVAL_BATCH_SIZE),
    "--grad_accum", str(GRAD_ACCUM), "--lr", str(LEARNING_RATE), "--warmup_steps", "20",
    "--max_length", str(MAX_LENGTH), "--logging_steps", "5",
    "--lora_r", str(LORA_R), "--lora_alpha", str(LORA_ALPHA), "--lora_dropout", str(LORA_DROPOUT),
    "--eval_strategy", "epoch", "--save_strategy", "epoch", "--save_total_limit", "2",
    "--early_stopping", "--early_stopping_patience", str(EARLY_STOPPING_PATIENCE),
    "--early_stopping_threshold", "0.0",
]
print(subprocess.list2cmdline([str(part) for part in TRAIN_COMMAND]))
experiment = {
    "run_id": RUN_ID, "python": sys.executable, "package_versions": package_versions,
    "dataset_manifest": str(EXPORT_DIR / "dataset_manifest.json"),
    "training_command": [str(part) for part in TRAIN_COMMAND],
    "script_sha256": {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                      for p in (TRAIN_SCRIPT, EVAL_SCRIPT, INFER_SCRIPT)},
    "evaluation": {"img_size": IMG_SIZE, "max_new_tokens": MAX_NEW_TOKENS,
                   "temperature": TEMPERATURE, "min_p": MIN_P, "seed": SEED},
    "run_path_override": str(RUN_PATH_OVERRIDE) if RUN_PATH_OVERRIDE else None,
}
ATTEMPT_DIR.mkdir(parents=True, exist_ok=True)
(ATTEMPT_DIR / "experiment.json").write_text(json.dumps(experiment, indent=2), encoding="utf-8")

c:\Users\moht2\.conda\envs\afterimage_env\python.exe -u c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\train_dr_qwen3vl_runsave.py --csv_path c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\aptos_messidor\data\472ed7fde07b97fcb0b4\train.csv --val_csv c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\aptos_messidor\data\472ed7fde07b97fcb0b4\val.csv --test_csv c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\aptos_messidor\data\472ed7fde07b97fcb0b4\test.csv --image_dir C:\Users\moht2\Downloads\DR_Aptos_Messidor_Only_Full_Dataset\Res_512\Aptos_messidor_external_CLAHE --ext .png --img_size 512 --seed 42 --model_name unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit --load_in_4bit --output_dir c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\aptos_messidor\experiments\20261104T153559Z_bc11a971\checkpoints --runs_root c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\aptos_messidor\experiments\20261104T153559Z_bc11a971\ru

3174

In [6]:
if RUN_PATH_OVERRIDE is None:
    if CHECKPOINT_DIR.exists() or RUNS_ROOT.exists():
        raise FileExistsError("This attempt already has training output. Use a new RUN_ID or set RUN_PATH_OVERRIDE.")
    run_live(TRAIN_COMMAND, LOG_DIR / "training.log")
else:
    print("Training skipped; evaluating the explicitly selected run:", RUN_PATH_OVERRIDE)

c:\Users\moht2\.conda\envs\afterimage_env\python.exe -u c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\train_dr_qwen3vl_runsave.py --csv_path c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\aptos_messidor\data\472ed7fde07b97fcb0b4\train.csv --val_csv c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\aptos_messidor\data\472ed7fde07b97fcb0b4\val.csv --test_csv c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\aptos_messidor\data\472ed7fde07b97fcb0b4\test.csv --image_dir C:\Users\moht2\Downloads\DR_Aptos_Messidor_Only_Full_Dataset\Res_512\Aptos_messidor_external_CLAHE --ext .png --img_size 512 --seed 42 --model_name unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit --load_in_4bit --output_dir c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\aptos_messidor\experiments\20261104T153559Z_bc11a971\checkpoints --runs_root c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\aptos_messidor\experiments\20261104T153559Z_bc11a971\ru

In [7]:
if RUN_PATH_OVERRIDE is not None:
    RUN_PATH = Path(RUN_PATH_OVERRIDE).expanduser().resolve()
else:
    candidates = list(RUNS_ROOT.glob("*/run_info.json"))
    if len(candidates) != 1:
        raise RuntimeError(f"Expected exactly one completed run for this attempt; found {len(candidates)}.")
    RUN_PATH = candidates[0].parent.resolve()
LORA_DIR = RUN_PATH if (RUN_PATH / "adapter_config.json").is_file() else RUN_PATH / "best_lora"
if not (LORA_DIR / "adapter_config.json").is_file():
    raise FileNotFoundError("No adapter_config.json in " + str(LORA_DIR))
if not any((LORA_DIR / name).is_file() and (LORA_DIR / name).stat().st_size > 0
           for name in ("adapter_model.safetensors", "adapter_model.bin")):
    raise FileNotFoundError("Missing non-empty adapter weights in " + str(LORA_DIR))
run_info_path = LORA_DIR.parent / "run_info.json"
run_info = json.loads(run_info_path.read_text(encoding="utf-8")) if run_info_path.is_file() else {}
if run_info:
    if run_info["model_name"] != BASE_MODEL or str(run_info["img_size"]) != str(IMG_SIZE):
        raise ValueError("BASE_MODEL / IMG_SIZE differ from the selected run_info.json. Correct configuration first.")
    display(pd.Series({key: run_info.get(key) for key in ("best_epoch", "best_step", "best_eval_loss", "final_run_dir")}))
(ATTEMPT_DIR / "selected_run.json").write_text(
    json.dumps({"run_path": str(RUN_PATH), "lora_dir": str(LORA_DIR), "run_info": run_info}, indent=2), encoding="utf-8")
print("Selected adapter:", LORA_DIR)

best_epoch                                                      6.0
best_step                                                      2202
best_eval_loss                                             0.006926
final_run_dir     c:\Users\moht2\Documents\Afterimage\finetuning...
dtype: object

Selected adapter: C:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\aptos_messidor\experiments\20261104T153559Z_bc11a971\runs\aptos_external_messidor_20261104T153559Z_bc11a971_lr0.0005_epoch6_batchsize8\best_lora


## 5. Test all held-out images

`--limit 0` evaluates the entire supplied test CSV. Temperature 0 uses greedy decoding. The optional
smoke check below is a three-image diagnostic and is never used for the final reports.
The evaluator loads the selected adapter once and shares `predict_one` / answer parsing with
`eval_infer_dr_qwen3vl.py`. Each process exits before the next one starts, releasing its GPU memory.

In [8]:
def evaluation_command(csv_path, output_csv, limit=0):
    return [
        sys.executable, "-u", EVAL_SCRIPT, "--run_path", RUN_PATH, "--base_model", BASE_MODEL,
        "--load_in_4bit", "--csv_path", csv_path, "--image_dir", IMAGE_DIR, "--ext", ".png",
        "--img_size", str(IMG_SIZE), "--max_new_tokens", str(MAX_NEW_TOKENS),
        "--temperature", str(TEMPERATURE), "--min_p", str(MIN_P), "--seed", str(SEED),
        "--limit", str(limit), "--save_preds_csv", output_csv,
    ]

if RUN_EVAL_SMOKE:
    smoke_csv = REPORT_DIR / "smoke_3_images.raw.csv"
    run_live(evaluation_command(CSV_FILES["test"], smoke_csv, limit=3), LOG_DIR / "evaluation_smoke.log")
    display(pd.read_csv(smoke_csv)[["image_path", "y_true", "y_pred", "parse_ok"]])
else:
    print("Optional smoke evaluation skipped.")

Optional smoke evaluation skipped.


In [9]:
RAW_PREDICTIONS = REPORT_DIR / "test_all.raw.csv"
run_live(evaluation_command(CSV_FILES["test"], RAW_PREDICTIONS, limit=0), LOG_DIR / "evaluation_test_all.log")
print("Full test predictions:", RAW_PREDICTIONS)

c:\Users\moht2\.conda\envs\afterimage_env\python.exe -u c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\eval_dataset_dr_qwen3vl_selectrun.py --run_path C:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\aptos_messidor\experiments\20261104T153559Z_bc11a971\runs\aptos_external_messidor_20261104T153559Z_bc11a971_lr0.0005_epoch6_batchsize8 --base_model unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit --load_in_4bit --csv_path c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\aptos_messidor\data\472ed7fde07b97fcb0b4\test.csv --image_dir C:\Users\moht2\Downloads\DR_Aptos_Messidor_Only_Full_Dataset\Res_512\Aptos_messidor_external_CLAHE --ext .png --img_size 512 --max_new_tokens 48 --temperature 0.0 --min_p 0.1 --seed 42 --limit 0 --save_preds_csv c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\aptos_messidor\experiments\20261104T153559Z_bc11a971\reports\test_all.raw.csv
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1104 21:

## 6. Save combined, APTOS-only, and external Messidor reports

All 1,424 predictions must match the held-out CSV exactly before reporting. Separate source reports
are partitions of this same inference pass. Each report includes strict accuracy, quadratic weighted
kappa (QWK), macro F1, balanced accuracy, per-class precision/recall/F1, a 5-by-5 confusion matrix,
parse-failure count and the full per-image output.

**Parse failures:** for compatibility, the evaluator substitutes grade 0 when an answer cannot be
parsed. `strict_accuracy` always counts those cases as wrong. QWK, F1, balanced accuracy, and confusion
matrices use the grade-0 fallback; inspect `bad_parses` alongside them. Undefined QWK is saved as JSON
`null` rather than a misleading score. Class metrics use all five grades, with zero for undefined
precision/recall. Balanced accuracy averages recall over the grades present in that test subset.

In [10]:
LABEL_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]

def attach_test_metadata(raw_csv, expected):
    predictions = pd.read_csv(raw_csv)
    needed = {"image_path", "y_true", "y_pred", "parse_ok", "raw_output"}
    if not needed.issubset(predictions.columns):
        raise ValueError("Prediction file is missing columns: " + str(needed - set(predictions.columns)))
    if len(predictions) != len(expected):
        raise ValueError(f"Expected {len(expected)} predictions, got {len(predictions)}")
    predictions["image_path"] = predictions["image_path"].map(lambda p: str(Path(p).resolve()))
    predictions["parse_ok"] = predictions["parse_ok"].astype(str).str.lower().map({"true": True, "false": False})
    if predictions["parse_ok"].isna().any():
        raise ValueError("Invalid parse_ok flags.")
    merged = expected.merge(predictions, on="image_path", how="outer", validate="one_to_one", indicator=True)
    if not merged["_merge"].eq("both").all() or not merged["label"].eq(merged["y_true"]).all():
        raise ValueError("Predictions do not match the held-out paths/labels.")
    if not merged["y_pred"].isin(range(5)).all():
        raise ValueError("Prediction grades must be 0..4 after evaluator fallback.")
    return merged.drop(columns="_merge")

def calculate_metrics(frame):
    truth = frame["y_true"].to_numpy(dtype=int)
    pred = frame["y_pred"].to_numpy(dtype=int)
    cm = np.zeros((5, 5), dtype=int)
    np.add.at(cm, (truth, pred), 1)
    true_count, pred_count = cm.sum(axis=1), cm.sum(axis=0)
    precision = np.divide(cm.diagonal(), pred_count, out=np.zeros(5), where=pred_count != 0)
    recall = np.divide(cm.diagonal(), true_count, out=np.zeros(5), where=true_count != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(5), where=(precision + recall) != 0)
    weights = (np.arange(5)[:, None] - np.arange(5)[None, :]) ** 2 / 16.0
    expected = np.outer(true_count, pred_count) / len(frame)
    denominator = float((weights * expected).sum())
    qwk = float(1 - (weights * cm).sum() / denominator) if denominator > 0 else None
    return {
        "samples": len(frame), "bad_parses": int((~frame["parse_ok"]).sum()),
        "strict_accuracy": float(((truth == pred) & frame["parse_ok"].to_numpy(dtype=bool)).mean()),
        "accuracy_with_grade0_fallback": float((truth == pred).mean()),
        "qwk": qwk, "macro_f1": float(f1.mean()),
        "balanced_accuracy": float(recall[true_count > 0].mean()),
        "confusion_matrix": cm.tolist(),
        "per_class": [{"grade": i, "label": LABEL_NAMES[i], "support": int(true_count[i]),
                       "precision": float(precision[i]), "recall": float(recall[i]), "f1": float(f1[i])} for i in range(5)],
    }

def save_report(name, frame, prediction_origin):
    report = {
        "name": name, "run_path": str(RUN_PATH), "lora_dir": str(LORA_DIR),
        "source_csv_sha256": source_sha256, "split_policy": manifest["split_policy"],
        "prediction_origin": str(prediction_origin),
        "invalid_prediction_policy": "grade_0_fallback; strict_accuracy counts every parse failure wrong",
        "evaluation_config": experiment["evaluation"], **calculate_metrics(frame),
    }
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    frame.to_csv(REPORT_DIR / f"{name}.predictions.csv", index=False)
    (REPORT_DIR / f"{name}.summary.json").write_text(json.dumps(report, indent=2, allow_nan=False), encoding="utf-8")
    class_table = pd.DataFrame(report["per_class"])
    class_table.to_csv(REPORT_DIR / f"{name}.per_class.csv", index=False)
    cm_table = pd.DataFrame(report["confusion_matrix"], index=pd.Index(range(5), name="true_grade"), columns=range(5))
    cm_table.to_csv(REPORT_DIR / f"{name}.confusion_matrix.csv")
    print(name)
    display(cm_table)
    display(class_table)
    return report

full_predictions = attach_test_metadata(RAW_PREDICTIONS, frames["test"])
reports = []
for name, subset in (
    ("test_all", full_predictions),
    ("test_aptos", full_predictions.loc[full_predictions["source_dataset"].eq("aptos")]),
    ("test_messidor_external", full_predictions.loc[full_predictions["source_dataset"].eq("messidor")]),
):
    reports.append(save_report(name, subset, RAW_PREDICTIONS))
summary_columns = ["name", "samples", "bad_parses", "strict_accuracy", "qwk", "macro_f1", "balanced_accuracy"]
summary = pd.DataFrame(reports)[summary_columns]
summary.to_csv(REPORT_DIR / "test_comparison.csv", index=False)
display(summary)
assert reports[0]["samples"] == reports[1]["samples"] + reports[2]["samples"] == len(frames["test"])
print("Reports saved to:", REPORT_DIR)

test_all


,0,1,2,3,4
true_grade,,,,,
0,643,4,1,0,0
1,213,22,4,0,1
2,170,99,116,1,2
3,3,11,64,12,2
4,4,4,24,6,18


,grade,label,support,precision,recall,f1
0,0,No DR,648,0.622459,0.992284,0.765021
1,1,Mild,240,0.157143,0.091667,0.115789
2,2,Moderate,388,0.555024,0.298969,0.388610
3,3,Severe,92,0.631579,0.130435,0.216216
4,4,Proliferative DR,56,0.782609,0.321429,0.455696


test_aptos


,0,1,2,3,4
true_grade,,,,,
0,176,3,1,0,0
1,7,21,4,0,1
2,2,16,78,1,1
3,0,0,12,8,1
4,0,1,16,4,14


,grade,label,support,precision,recall,f1
0,0,No DR,180,0.951351,0.977778,0.964384
1,1,Mild,33,0.512195,0.636364,0.567568
2,2,Moderate,98,0.702703,0.795918,0.746411
3,3,Severe,21,0.615385,0.380952,0.470588
4,4,Proliferative DR,35,0.823529,0.400000,0.538462


test_messidor_external


,0,1,2,3,4
true_grade,,,,,
0,467,1,0,0,0
1,206,1,0,0,0
2,168,83,38,0,1
3,3,11,52,4,1
4,4,3,8,2,4


,grade,label,support,precision,recall,f1
0,0,No DR,468,0.550708,0.997863,0.709726
1,1,Mild,207,0.010101,0.004831,0.006536
2,2,Moderate,290,0.387755,0.131034,0.195876
3,3,Severe,71,0.666667,0.056338,0.103896
4,4,Proliferative DR,21,0.666667,0.190476,0.296296


,name,samples,bad_parses,strict_accuracy,qwk,macro_f1,balanced_accuracy
0,test_all,1424,0,0.569522,0.613971,0.388266,0.366957
1,test_aptos,367,0,0.809264,0.875949,0.657482,0.638202
2,test_messidor_external,1057,0,0.486282,0.469845,0.262466,0.276109


Reports saved to: c:\Users\moht2\Documents\Afterimage\finetuning\Qwen3\artifacts\aptos_messidor\experiments\20261104T153559Z_bc11a971\reports


## 7. Optional independent APTOS / Messidor inference runs

Enable `RUN_SEPARATE_SOURCE_EVAL` in configuration only when you want the evaluator to load the adapter
again for each source CSV. This repeats inference for all test images and saves separately named outputs;
the combined/source reports above are already complete without this step.

In [11]:
if RUN_SEPARATE_SOURCE_EVAL:
    independent_reports = []
    for key, report_name in (("test_aptos", "independent_test_aptos"), ("test_messidor", "independent_test_messidor_external")):
        raw_csv = REPORT_DIR / f"{report_name}.raw.csv"
        run_live(evaluation_command(CSV_FILES[key], raw_csv, limit=0), LOG_DIR / f"{report_name}.log")
        subset = attach_test_metadata(raw_csv, frames[key])
        independent_reports.append(save_report(report_name, subset, raw_csv))
    display(pd.DataFrame(independent_reports)[summary_columns])
else:
    print("Extra inference passes skipped. Separate source reports were saved in section 6.")

Extra inference passes skipped. Separate source reports were saved in section 6.


## Saved artifacts and reproducibility

Everything generated by this notebook lives under the ignored `finetuning/Qwen3/artifacts/` directory:

- `data/<fingerprint>/`: unchanged-split train/validation/test CSV exports, source-specific test CSVs,
  original CSV hash, export hashes and row counts.
- `experiments/<RUN_ID>/experiment.json`: exact command, script hashes, Python executable and package versions.
- `experiments/<RUN_ID>/checkpoints/`: trainer checkpoints, kept to the most recent two.
- `experiments/<RUN_ID>/runs/<run_name>/best_lora/` and `run_info.json`: selected adapter/processor and validation history.
- `experiments/<RUN_ID>/reports/`: raw answers, combined/source predictions, summaries, per-class metrics,
  confusion matrices and `test_comparison.csv`.
- `experiments/<RUN_ID>/logs/`: streamed logs and exact subprocess argument lists.

Greedy generation and a fixed seed improve repeatability; GPU kernels and changes in model revisions,
drivers or package versions can still change results. Keep the complete run directory outside Git.
The recorded script and data hashes help identify what was run; Hub model revisions are not pinned here.

API references: [Unsloth vision fine-tuning](https://unsloth.ai/docs/basics/vision-fine-tuning) and
[TRL SFTTrainer](https://huggingface.co/docs/trl/sft_trainer). Compatibility was also checked against
the installed TRL 1.15.0 / Transformers 5.17.0 / Unsloth 2026.10.10 source in `afterimage_env`.